In [10]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [11]:
df = pd.read_csv('./data/dataset.csv', parse_dates=['Date']).set_index('Date').sort_index()
df.head(3)

,BAA10Y,DFF,DGS10,DGS2,CPIAUCSL,CPIAUCSL_days_since,PPIACO,PPIACO_days_since,PCEPI,PCEPI_days_since,...,RSAFS,RSAFS_days_since,UMCSENT,UMCSENT_days_since,TCU,TCU_days_since,DTWEXBGS,DCOILWTICO,DEXUSEU,EURUSD_FX_Vol
Date,,,,,,,,,,,,,,,,,,,,,
1990-01-01,NaN,7.97,NaN,NaN,127.5,0.0,114.9,0.0,58.553,0.0,...,NaN,NaN,93.0,0.0,82.4179,0.0,NaN,NaN,NaN,NaN
1990-01-02,1.91,8.54,7.94,7.87,127.5,1.0,114.9,1.0,58.553,1.0,...,NaN,NaN,93.0,1.0,82.4179,1.0,NaN,22.88,NaN,NaN
1990-01-03,1.87,8.37,7.99,7.94,127.5,2.0,114.9,2.0,58.553,2.0,...,NaN,NaN,93.0,2.0,82.4179,2.0,NaN,23.81,NaN,NaN


In [12]:
# Forecast horizon (days ahead). Change this to forecast further out.
H = 1

# Supervised setup: features at time t predict BAA10Y at time t+H.
# Shifting by -H aligns each row's target with the future value.
y_future = df['BAA10Y'].shift(-H)
y_now    = df['BAA10Y']
X_full   = df.drop(columns=['BAA10Y'])

# Drop rows where either the current spread (needed for naive baseline)
# or the future spread (the target) is NaN.
mask = y_future.notna() & y_now.notna()
X = X_full.loc[mask]
y = y_future.loc[mask]
y_t = y_now.loc[mask]  # current spread, kept for the naive last-value baseline

print(f"Horizon H = {H} day(s)")
print(f"Samples: {len(y)}  ({y.index.min().date()} → {y.index.max().date()})")

Horizon H = 1 day(s)
Samples: 13148  (1990-01-02 → 2025-12-31)


In [13]:
# Chronological split — NEVER shuffle time-series data.
# Using fractions so it adapts as the dataset grows.
n = len(y)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, X_val, X_test = X.iloc[:train_end], X.iloc[train_end:val_end], X.iloc[val_end:]
y_train, y_val, y_test = y.iloc[:train_end], y.iloc[train_end:val_end], y.iloc[val_end:]
y_t_val,  y_t_test     = y_t.iloc[train_end:val_end], y_t.iloc[val_end:]

print(f"train: {y_train.index.min().date()} → {y_train.index.max().date()}  ({len(y_train)} rows)")
print(f"val:   {y_val.index.min().date()} → {y_val.index.max().date()}  ({len(y_val)} rows)")
print(f"test:  {y_test.index.min().date()} → {y_test.index.max().date()}  ({len(y_test)} rows)")

train: 1990-01-02 → 2015-03-14  (9203 rows)
val:   2015-03-15 → 2020-08-06  (1972 rows)
test:  2020-08-07 → 2025-12-31  (1973 rows)


In [14]:
# Baselines. None of these use X — they exist to establish a floor that any
# real model must beat.
#
#   constant_mean   : predict y_train.mean() for every row
#   constant_median : predict y_train.median() for every row
#   naive           : predict BAA10Y[t+H] = BAA10Y[t]   (very strong on persistent series)

def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"  {name:18s}  MAE={mae:.4f}  RMSE={rmse:.4f}")

mean_val   = np.full(len(y_val),  y_train.mean())
mean_test  = np.full(len(y_test), y_train.mean())
median_val  = np.full(len(y_val),  y_train.median())
median_test = np.full(len(y_test), y_train.median())
naive_val   = y_t_val.values
naive_test  = y_t_test.values

print("Validation:")
evaluate("constant_mean",   y_val, mean_val)
evaluate("constant_median", y_val, median_val)
evaluate("naive",           y_val, naive_val)

print("\nTest:")
evaluate("constant_mean",   y_test, mean_test)
evaluate("constant_median", y_test, median_test)
evaluate("naive",           y_test, naive_test)

Validation:
  constant_mean       MAE=0.4134  RMSE=0.5191
  constant_median     MAE=0.4154  RMSE=0.5636
  naive               MAE=0.0125  RMSE=0.0247

Test:
  constant_mean       MAE=0.4810  RMSE=0.5419
  constant_median     MAE=0.3661  RMSE=0.4281
  naive               MAE=0.0147  RMSE=0.0251
